<a href="https://colab.research.google.com/github/shijithpulikkal/CodingFactory/blob/main/CF9%20Customer%20Retention%20Rate%20(month-over-month).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')
df= pd.read_excel('/content/drive/My Drive/Colab Notebooks/Superstore.xlsx')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Sales_Month'] = df['Order_Date'].dt.to_period('M')
df.head()

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit,Sales_Month
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016-11
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016-11
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016-06
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015-10
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015-10


In [4]:
monthly_customers = df[['Sales_Month', 'Customer_ID']].drop_duplicates()
monthly_customers = monthly_customers.sort_values(['Sales_Month', 'Customer_ID'])
monthly_customers

,Sales_Month,Customer_ID
865,2014-01,AJ-10780
4937,2014-01,BD-11605
6474,2014-01,BS-11590
8149,2014-01,CA-11965
763,2014-01,CS-12250
...,...,...
2851,2017-12,VF-21715
1752,2017-12,VW-21775
9409,2017-12,WB-21850
5474,2017-12,YC-21895


In [5]:
monthly_customers['PrevPurchaseMonth'] = monthly_customers.groupby('Customer_ID')['Sales_Month'].shift(1)
monthly_customers

,Sales_Month,Customer_ID,PrevPurchaseMonth
865,2014-01,AJ-10780,NaT
4937,2014-01,BD-11605,NaT
6474,2014-01,BS-11590,NaT
8149,2014-01,CA-11965,NaT
763,2014-01,CS-12250,NaT
...,...,...,...
2851,2017-12,VF-21715,2017-07
1752,2017-12,VW-21775,2017-11
9409,2017-12,WB-21850,2017-11
5474,2017-12,YC-21895,2016-09


In [6]:
monthly_customers['MonthDiff'] = (
    monthly_customers['Sales_Month'].astype(int)
    - monthly_customers['PrevPurchaseMonth']
        .fillna(monthly_customers['Sales_Month'])
        .astype(int)
)

In [8]:
monthly_customers['IsRetained'] = (
    monthly_customers['MonthDiff'] == 1
).astype(int)

In [9]:
retention = (monthly_customers.groupby('Sales_Month').agg(TotalCustomers=('Customer_ID', 'count'),RetainedCustomers = ('IsRetained','sum')).reset_index())
retention

,Sales_Month,TotalCustomers,RetainedCustomers
0,2014-01,32,0
1,2014-02,27,3
2,2014-03,69,4
3,2014-04,64,4
4,2014-05,67,6
5,2014-06,63,5
6,2014-07,65,4
7,2014-08,70,7
8,2014-09,118,10
9,2014-10,75,14


In [10]:
retention['RetentionRatePct'] = (retention['RetainedCustomers']/retention['TotalCustomers'] * 100).round(2)
retention

,Sales_Month,TotalCustomers,RetainedCustomers,RetentionRatePct
0,2014-01,32,0,0.00
1,2014-02,27,3,11.11
2,2014-03,69,4,5.80
3,2014-04,64,4,6.25
4,2014-05,67,6,8.96
5,2014-06,63,5,7.94
6,2014-07,65,4,6.15
7,2014-08,70,7,10.00
8,2014-09,118,10,8.47
9,2014-10,75,14,18.67


In [11]:
retention['Sales_Month'] = (
    retention['Sales_Month']
      .dt.to_timestamp()
      .dt.strftime('%b %Y')
)
retention

,Sales_Month,TotalCustomers,RetainedCustomers,RetentionRatePct
0,Jan 2014,32,0,0.00
1,Feb 2014,27,3,11.11
2,Mar 2014,69,4,5.80
3,Apr 2014,64,4,6.25
4,May 2014,67,6,8.96
5,Jun 2014,63,5,7.94
6,Jul 2014,65,4,6.15
7,Aug 2014,70,7,10.00
8,Sep 2014,118,10,8.47
9,Oct 2014,75,14,18.67
